In [ ]:
import math
from collections import Counter #hashmap like set that keeps track over unique key feqs

def n_grams(seq, n):
    if type(seq) == str:
        seq = seq.split()
    return [tuple(seq[i:i+n]) for i in range(len(seq)-n+1)]

def GLEU(S, H, R, max_n = 4, _lambda = 1):    
    #tokenize on wite space
    tok_H = H.split() if isinstance(H, str) else H
    tok_R = R.split() if isinstance(R, str) else R
    #brevity penalty that pushed the model for producing shorter hypotheses 
    BP = 1 if len(tok_H) >= len(tok_R) else math.exp(1 - (len(tok_R)/len(tok_H)))
    
    precision_at_n = []
    
    for n in range(1, max_n +1): 
        Sg = n_grams(S, n)
        Hg = n_grams(H, n)
        Rg = n_grams(R, n)
        
        # convert n-gram collections to dict with freq counts as values
        CS, CH, CR = Counter(Sg), Counter(Hg), Counter(Rg)
                
        # n-grams present in the candidate that overlap with the reference but not the source (R \ S)
        count_R_S = CH & (CR - CS)
        
        # n-grams in the candidate that are in the source but not the reference (S \ R)
        count_S_R = CH & (CS - CR)
        
        # base overlap between candidate and reference
        count_R = CH & CR
        
        #n-gram freqs
        val_R_S = sum(count_R_S.values())
        val_S_R = sum(count_S_R.values())
        val_R = sum(count_R.values())
        
        denominator = sum(CH.values())
        
        if denominator > 0:
            # Apply Equation 1: (Base Overlap + Reward - Penalty) / Denominator
            numerator = val_R + val_R_S - (_lambda * val_S_R)
            
            # Clip at near zero to prevent negative precision, use 1e-7 to prevent devide by zero error. 
            precision = max(1e-7, numerator) / denominator
            precision_at_n.append(precision)
        else:
            precision_at_n.append(1e-7)
    
    geometric_mean = math.exp(sum(math.log(p) for p in precision_at_n)/max_n)
    
    return geometric_mean * BP

In [ ]:
# from collections import Counter
# #Goal: candidates (C) that overlap with the reference (R) but not the source (S)
# def first_term(R,S,C):
#     #R\S & C  correctly added n-grams
#     return C & (R - S)

# #λ (CountS\R(n-gram))
# def second_term(R,S,C): 
#     # S\R & C  incorrectly added n-grams
#     return C & (S - R)
    

# def third_term(R,C): 
#     # R&C  correct n-gram overlap between prediction and ref
#     return R & C


# def denominator():
#     pass

first term: Counter({'apple': 2}), second term Counter({'banana': 3}), third_term: Counter({'apple': 2, 'banana': 2})


In [ ]:
C = "the cat sat on the mat".split(" ")

# Multiset difference is needed here. If A has two "cats" and B has one, A \ B should have one "cat".
def in_A_not_B(A, B):
    result = list(A)
    for item in B:
        if item in result:
            result.remove(item)
    return result

#eq 2 and 3 
def count(bag, n_gram):
    c = 0 
    for n_gram_prime in bag: 
        if n_gram_prime == n_gram:
            c += 1
    return c 

#eq 1 
def N_gram_Precision(S, R, C):
    R_S = in_A_not_B(R, S)
    S_R = in_A_not_B(S, R)
    
    #enumerator  
    enumerator = 0 
    lam = 1
    for n_gram in set(C): 
         enumerator += count(R_S, n_gram) - (lam * count(S_R, n_gram)) + count(R, n_gram)
         
    #denominator
    denominator = len(C) + len(R_S)

    if denominator == 0:
        return 1e-9
        
    pn = (enumerator / denominator) + 1e-9
    
    return pn


import math



def get_ngrams(tokens, n):
    return [" ".join(tokens[i:i+n] for i in range(len(tokens)-n+1))]



#eq.4 & 5
def GLEU(C,R,S,N, weights = None):
    
    N = list(range(1,N+1))
    
    #Brevity Penalty
    BP = 1 if len(C) > len(R) else math.exp(1-(len(R)/len(C)))
    BP = 0 if len(C) == 0 else BP
    
    weights = [1.0 / len(N)] * len(N)

GLEU(1,1,1,4)

[1, 2, 3, 4]


In [ ]:
import math

# Helper to group words into n-grams of size 'n'
def get_ngrams(tokens, n):
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

# Eq 4 and 5
def GLEU(C, R, S, N=[1, 2, 3, 4], weights=None):
    c = len(C)
    r = len(R)
    
    # Equation 4: Brevity Penalty (BP)
    if c > r:
        BP = 1.0
    elif c == 0:
        BP = 0.0 # Catch division by zero
    else:
        BP = math.exp(1 - (r / c))
        
    # Default weights are uniform (e.g., 0.25 for each if N has 4 items)
    if weights is None:
        weights = [1.0 / len(N)] * len(N)
        
    sum_log_pn = 0
    
    # Equation 5: Summation loop
    for idx, n in enumerate(N):
        # Generate the n-grams for this specific loop iteration
        C_n = get_ngrams(C, n)
        R_n = get_ngrams(R, n)
        S_n = get_ngrams(S, n)
        
        # Calculate p'_n using your Equation 1 function
        pn = N_gram_Precision(S_n, R_n, C_n)
        
        # Apply the weight and log, then add to the sum
        w_n = weights[idx]
        sum_log_pn += w_n * math.log(pn)
        
    # Equation 5: Final calculation
    return BP * math.exp(sum_log_pn)

In [ ]:
A = [1,2,3,3]
B = [3,4]

in_A_not_B(A,B)

[1, 2, 3]